# EDA with SQL — required rubric queries
The IBM SQL assignment historically uses a separate `SPACEXTBL` dataset containing `Mission_Outcome`, `Landing_Outcome`, `Booster_Version`, etc. Download it independently to preserve the official query semantics.

In [1]:
import pandas as pd, sqlite3
from pathlib import Path
D=Path('data');O=Path('outputs');O.mkdir(exist_ok=True)
df=pd.read_csv(D/'Spacex.csv')
print(df.columns.tolist());display(df.head())
conn=sqlite3.connect(':memory:');df.to_sql('SPACEXTBL',conn,index=False,if_exists='replace')

['Date', 'Time (UTC)', 'Booster_Version', 'Launch_Site', 'Payload', 'PAYLOAD_MASS__KG_', 'Orbit', 'Customer', 'Mission_Outcome', 'Landing_Outcome']


,Date,Time (UTC),Booster_Version,Launch_Site,Payload,PAYLOAD_MASS__KG_,Orbit,Customer,Mission_Outcome,Landing_Outcome
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt


101

In [2]:
# Column names differ slightly between published copies: normalize names to SQL-friendly keys.
cols={c:''.join(ch if ch.isalnum() else '_' for ch in c.strip()).strip('_').upper() for c in df.columns}
df=df.rename(columns=cols);df.to_sql('SPACEXTBL',conn,index=False,if_exists='replace')
print('SQL columns:',df.columns.tolist())
def query(sql,label):
    print('\n'+label+'\n'+sql);result=pd.read_sql_query(sql,conn);display(result)
    result.to_csv(O/('sql_'+str(len(list(O.glob('sql_*.csv')))+1).zfill(2)+'.csv'),index=False)
    return result

SQL columns: ['DATE', 'TIME__UTC', 'BOOSTER_VERSION', 'LAUNCH_SITE', 'PAYLOAD', 'PAYLOAD_MASS__KG', 'ORBIT', 'CUSTOMER', 'MISSION_OUTCOME', 'LANDING_OUTCOME']


In [3]:
# Standard IBM capstone SQL queries. Review printed schema before running if column labels differ.
queries=[
('Launch site names',"SELECT DISTINCT LAUNCH_SITE FROM SPACEXTBL"),
('CCA launches',"SELECT * FROM SPACEXTBL WHERE LAUNCH_SITE LIKE 'CCA%' LIMIT 5"),
('NASA payload',"SELECT SUM(PAYLOAD_MASS__KG_) AS TOTAL_KG FROM SPACEXTBL WHERE CUSTOMER LIKE '%NASA%'"),
('Average F9 v1.1 payload',"SELECT AVG(PAYLOAD_MASS__KG_) AS AVG_KG FROM SPACEXTBL WHERE BOOSTER_VERSION LIKE 'F9 v1.1%'"),
('First ground pad landing',"SELECT MIN(DATE) AS FIRST_LANDING FROM SPACEXTBL WHERE LANDING_OUTCOME LIKE 'Success (ground pad)%'"),
('Drone ship landings, 4–6t',"SELECT DISTINCT BOOSTER_VERSION FROM SPACEXTBL WHERE LANDING_OUTCOME LIKE 'Success (drone ship)%' AND PAYLOAD_MASS__KG_ > 4000 AND PAYLOAD_MASS__KG_ < 6000"),
('Mission outcomes',"SELECT MISSION_OUTCOME,COUNT(*) AS N FROM SPACEXTBL GROUP BY MISSION_OUTCOME"),
('Maximum payload boosters',"SELECT DISTINCT BOOSTER_VERSION FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_=(SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL)"),
('2015 failed drone ship landings',"SELECT DATE,BOOSTER_VERSION,LAUNCH_SITE,LANDING_OUTCOME FROM SPACEXTBL WHERE DATE LIKE '2015%' AND LANDING_OUTCOME LIKE 'Failure (drone ship)%'"),
('Landing outcome ranking',"SELECT LANDING_OUTCOME,COUNT(*) AS N FROM SPACEXTBL WHERE DATE BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY LANDING_OUTCOME ORDER BY N DESC")]
for label,sql in queries:
    try:query(sql,label)
    except Exception as e:print('Query needs column/schema adjustment:',label,e)


Launch site names
SELECT DISTINCT LAUNCH_SITE FROM SPACEXTBL


,LAUNCH_SITE
0,CCAFS LC-40
1,VAFB SLC-4E
2,KSC LC-39A
3,CCAFS SLC-40



CCA launches
SELECT * FROM SPACEXTBL WHERE LAUNCH_SITE LIKE 'CCA%' LIMIT 5


,DATE,TIME__UTC,BOOSTER_VERSION,LAUNCH_SITE,PAYLOAD,PAYLOAD_MASS__KG,ORBIT,CUSTOMER,MISSION_OUTCOME,LANDING_OUTCOME
0,2010-06-04,18:45:00,F9 v1.0 B0003,CCAFS LC-40,Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success,Failure (parachute)
1,2010-12-08,15:43:00,F9 v1.0 B0004,CCAFS LC-40,"Dragon demo flight C1, two CubeSats, barrel of...",0,LEO (ISS),NASA (COTS) NRO,Success,Failure (parachute)
2,2012-05-22,7:44:00,F9 v1.0 B0005,CCAFS LC-40,Dragon demo flight C2,525,LEO (ISS),NASA (COTS),Success,No attempt
3,2012-10-08,0:35:00,F9 v1.0 B0006,CCAFS LC-40,SpaceX CRS-1,500,LEO (ISS),NASA (CRS),Success,No attempt
4,2013-03-01,15:10:00,F9 v1.0 B0007,CCAFS LC-40,SpaceX CRS-2,677,LEO (ISS),NASA (CRS),Success,No attempt



NASA payload
SELECT SUM(PAYLOAD_MASS__KG_) AS TOTAL_KG FROM SPACEXTBL WHERE CUSTOMER LIKE '%NASA%'
Query needs column/schema adjustment: NASA payload Execution failed on sql 'SELECT SUM(PAYLOAD_MASS__KG_) AS TOTAL_KG FROM SPACEXTBL WHERE CUSTOMER LIKE '%NASA%'': no such column: PAYLOAD_MASS__KG_

Average F9 v1.1 payload
SELECT AVG(PAYLOAD_MASS__KG_) AS AVG_KG FROM SPACEXTBL WHERE BOOSTER_VERSION LIKE 'F9 v1.1%'
Query needs column/schema adjustment: Average F9 v1.1 payload Execution failed on sql 'SELECT AVG(PAYLOAD_MASS__KG_) AS AVG_KG FROM SPACEXTBL WHERE BOOSTER_VERSION LIKE 'F9 v1.1%'': no such column: PAYLOAD_MASS__KG_

First ground pad landing
SELECT MIN(DATE) AS FIRST_LANDING FROM SPACEXTBL WHERE LANDING_OUTCOME LIKE 'Success (ground pad)%'


...

,MISSION_OUTCOME,N
0,Failure (in flight),1
1,Success,98
2,Success,1
3,Success (payload status unclear),1



Maximum payload boosters
SELECT DISTINCT BOOSTER_VERSION FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_=(SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL)
Query needs column/schema adjustment: Maximum payload boosters Execution failed on sql 'SELECT DISTINCT BOOSTER_VERSION FROM SPACEXTBL WHERE PAYLOAD_MASS__KG_=(SELECT MAX(PAYLOAD_MASS__KG_) FROM SPACEXTBL)': no such column: PAYLOAD_MASS__KG_

2015 failed drone ship landings
SELECT DATE,BOOSTER_VERSION,LAUNCH_SITE,LANDING_OUTCOME FROM SPACEXTBL WHERE DATE LIKE '2015%' AND LANDING_OUTCOME LIKE 'Failure (drone ship)%'


,DATE,BOOSTER_VERSION,LAUNCH_SITE,LANDING_OUTCOME
0,2015-01-10,F9 v1.1 B1012,CCAFS LC-40,Failure (drone ship)
1,2015-04-14,F9 v1.1 B1015,CCAFS LC-40,Failure (drone ship)



Landing outcome ranking
SELECT LANDING_OUTCOME,COUNT(*) AS N FROM SPACEXTBL WHERE DATE BETWEEN '2010-06-04' AND '2017-03-20' GROUP BY LANDING_OUTCOME ORDER BY N DESC


,LANDING_OUTCOME,N
0,No attempt,10
1,Success (drone ship),5
2,Failure (drone ship),5
3,Success (ground pad),3
4,Controlled (ocean),3
5,Uncontrolled (ocean),2
6,Failure (parachute),2
7,Precluded (drone ship),1
